# 01 — Phase 1 smoke test (local, Apple Silicon / MPS)

Goal: verify Gemma 4 E4B loads on this Mac, that the `_get_inner_base_model` unwrap pattern from gateguard works on Gemma 4's architecture, and that `last_hidden_state` looks sane despite Per-Layer Embeddings (PLE).

Runs in **bf16 on MPS** — no bitsandbytes. (nf4 4-bit quantization is CUDA-only; we'll do real QLoRA training in `02_train_colab.ipynb`.)

Tracks issue: https://github.com/Charlemagne-Labs/gateguard-suite/issues/73

## 1. Environment

In [1]:
import platform
import torch
import transformers

print(f"Python:       {platform.python_version()}")
print(f"Platform:     {platform.platform()}")
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"MPS available:    {torch.backends.mps.is_available()}")
print(f"MPS built:        {torch.backends.mps.is_built()}")

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Python:       3.11.8
Platform:     macOS-15.7.3-arm64-arm-64bit
torch:        2.11.0
transformers: 5.8.0
MPS available:    True
MPS built:        True
Using device: mps


## 2. Load Gemma 4 E4B in bf16

Uses **`google/gemma-4-E4B-it`** (instruction-tuned). Chosen to match the gateguard baseline `gemma-3-270m-it` so the F1 comparison varies only model size, not pretraining objective.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "google/gemma-4-E4B-it"  # instruction-tuned variant, matches gemma-3-270m-it baseline

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"  # mandatory for last-token pooling
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,  # `torch_dtype` is deprecated in transformers 5.x
    device_map={"": device},
)
model.eval()
print(f"Loaded {MODEL_ID}")

# Gemma 4's top-level config is composite (text + possibly vision).
# Real text-tower hyperparameters live under `text_config`. Fall back to the
# top-level config for older models that expose them directly.
cfg = getattr(model.config, "text_config", model.config)
print(f"config class:    {type(model.config).__name__}")
print(f"text-tower cfg:  {type(cfg).__name__}")
print(f"hidden_size:     {cfg.hidden_size}")
print(f"vocab_size:      {cfg.vocab_size}")
print(f"num_layers:      {cfg.num_hidden_layers}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Loaded google/gemma-4-E4B-it
config class:    Gemma4Config
text-tower cfg:  Gemma4TextConfig
hidden_size:     2560
vocab_size:      262144
num_layers:      42


## 3. Verify `_get_inner_base_model` unwrap

This is the lm_head bypass from `gateguard-suite/api_server/runners.py:156-175`. It MUST work on Gemma 4 — vocab is still 262K, so the wasted lm_head logits tensor is the same memory waste at the same scale.

Expected: `inner` is a Gemma 4 base module (likely `Gemma4Model` or similar), and `inner(...)` returns an object with `last_hidden_state` of shape `(B, T, hidden_size)`.

In [ ]:
def _get_inner_base_model(m):
    """Walks past PEFT wrappers, the CausalLM head, and any multimodal-style
    composite container to reach the text backbone (module that exposes `.layers`
    and whose forward returns `last_hidden_state`).

    Layouts handled:
      - PEFT:                     m.base_model.model -> ForCausalLM
      - Single-stack (Gemma 3):   ForCausalLM.model -> backbone with .layers
      - Multimodal (Gemma 4):     ForCausalLM.model -> Gemma4Model (vision +
                                  audio + language siblings) -> .language_model
                                  -> Gemma4TextModel with .layers
    """
    cur = m
    if hasattr(cur, "base_model") and hasattr(cur.base_model, "model"):
        cur = cur.base_model.model
    if hasattr(cur, "model"):
        cur = cur.model
    if not hasattr(cur, "layers"):
        for attr in ("language_model", "text_model"):
            if hasattr(cur, attr):
                cur = getattr(cur, attr)
                break
    return cur

inner = _get_inner_base_model(model)
print(f"inner type:        {type(inner).__name__}")
print(f"inner module path: {type(inner).__module__}")
print(f"inner has forward: {callable(getattr(inner, 'forward', None))}")
print(f"inner has layers:  {hasattr(inner, 'layers')}")
if hasattr(inner, 'layers'):
    print(f"num layers:        {len(inner.layers)}")

## 4. One forward through the inner model

Confirm that `inner(...)` returns `last_hidden_state` with the expected shape. If PLE breaks this — `last_hidden_state` missing or zeros — fall back to `output_hidden_states=True` on the full model and pull `hidden_states[-1]`.

In [ ]:
texts = [
    "https://example.com/login",
    "https://paypa1-secure.ru/verify-account.php",
]

batch = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
print(f"input_ids.shape: {batch['input_ids'].shape}")

with torch.inference_mode():
    out = inner(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
    )

lh = out.last_hidden_state
print(f"last_hidden_state.shape: {tuple(lh.shape)}")
print(f"last_hidden_state.dtype: {lh.dtype}")
print(f"any NaN:                 {torch.isnan(lh).any().item()}")
print(f"mean abs:                {lh.abs().mean().item():.4f}")

# Last-token pool (left-padded means the last token of each sequence is at index -1)
pooled = lh[:, -1, :]
print(f"pooled.shape:            {tuple(pooled.shape)}")

## 5. Sanity check: compare vs. `output_hidden_states=True` fallback

If step 4 worked, `inner(...).last_hidden_state` should equal `model(..., output_hidden_states=True).hidden_states[-1]`. If they diverge meaningfully, PLE is doing something we need to understand before relying on the unwrap.

_Note:_ this cell duplicates the forward pass — only run during the smoke test, never in production.

In [ ]:
with torch.inference_mode():
    out_full = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        output_hidden_states=True,
        use_cache=False,
    )

lh_fallback = out_full.hidden_states[-1]
diff = (lh - lh_fallback).abs().max().item()
print(f"max abs diff between inner.last_hidden_state and hidden_states[-1]: {diff:.6f}")

if diff < 1e-3:
    print("OK — unwrap is equivalent. Use `_get_inner_base_model` in production code.")
else:
    print("DIVERGENCE — investigate PLE before trusting the unwrap. Fall back to output_hidden_states=True for now.")

## 6. LoRA target module probe

Issue #73 suggests trying Gemma 3's set first: `["q_proj", "k_proj", "v_proj", "o_proj"]`. Confirm those names exist in the first decoder layer.

In [ ]:
# Walk the first transformer layer and list its named modules
layer0 = inner.layers[0] if hasattr(inner, "layers") else None
if layer0 is None:
    print("No `.layers` attribute on inner — model tree may differ. Print the top-level structure:")
    print(inner)
else:
    for name, _ in layer0.named_modules():
        if name and "proj" in name:
            print(name)

## Phase 1 acceptance

Sign off when all of these are green:

- [ ] Model loads on MPS in bf16 without OOM
- [ ] `_get_inner_base_model` returns a sensible Gemma 4 backbone module
- [ ] `inner(...).last_hidden_state` has shape `(B, T, hidden_size)`, no NaNs, non-trivial magnitude
- [ ] Diff vs. `output_hidden_states=True` fallback is < 1e-3 (or PLE divergence is understood)
- [ ] LoRA target modules `q_proj/k_proj/v_proj/o_proj` exist in layer 0

Once green, move to Phase 2 in `02_train_colab.ipynb` and start filling in `src/model.py` + `src/train.py`.